# Notebook 03 Draft Scratchpad

## Hypothesis Testing and Reliability Analysis

This scratchpad notebook is used to prototype the statistical analyses planned for Notebook 03. It contains trial window definitions, candidate tests, intermediate outputs, and rough comparisons. Only analyses that prove useful and interpretable here will be moved into the final clean notebook.

### Analysis Roadmap

**Setup**
- imports, data loading, variable selections, configuration

**MetroPT-3: pre-failure vs normal comparisons**
- pre-failure window definition
- normal-operation baseline definition
- Mann-Whitney U tests on selected continuous sensors

**Hydraulic system: condition-based comparisons**
- Mann-Whitney U / Kruskal-Wallis tests on selected feature-condition pairs
- stable vs unstable cycle comparisons

**Reliability metrics**
- MTBF and MTTR for both datasets

**Cross-dataset interpretation notes**
- comparison of statistically supported findings across shared physical concepts

**Keep / drop decisions for final Notebook 03**
- which results move to the clean notebook, which stay in scratchpad

## Setup

This section loads the processed MetroPT-3 and hydraulic datasets, imports the statistical tools required for hypothesis testing, and defines the initial variable selections carried forward from Notebook 02.

In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import mannwhitneyu, kruskal
from statsmodels.stats.multitest import multipletests

# Reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Load processed datasets
metro = pd.read_csv('../data/processed/metro_cleaned.csv', parse_dates=['timestamp'])
hydraulics = pd.read_csv('../data/processed/hydraulics_features.csv')

# Load MetroPT-3 documented failure events
metro_failures = pd.read_csv('../data/raw/metropt3_failure_reports.csv', parse_dates=['start_time', 'end_time'], keep_default_na=False)
metro_failures['duration'] = metro_failures['end_time'] - metro_failures['start_time']

print(f"MetroPT-3 shape: {metro.shape}")
print(f"Hydraulic shape: {hydraulics.shape}")
print(f"\nMetroPT-3 failure events: {len(metro_failures)}")
print(metro_failures)

# Variable selections guided by Notebook 02
metro_vars = ['TP2', 'H1', 'DV_pressure', 'Oil_temperature', 'Motor_current']

hydraulic_pairs = {
    'cooler_condition': ['TS1_mean', 'TS1_std'],
    'pump_leakage': ['EPS1_mean', 'PS1_mean'],
    'stable_flag': ['TS1_std', 'VS1_std', 'PS1_std', 'EPS1_std'],
}

MetroPT-3 shape: (1516948, 16)
Hydraulic shape: (2205, 73)

MetroPT-3 failure events: 4
   failure_id          start_time            end_time failure_type  \
0           1 2020-04-18 00:00:00 2020-04-18 23:59:00     Air leak   
1           2 2020-05-29 23:30:00 2020-05-30 06:00:00     Air leak   
2           3 2020-06-05 10:00:00 2020-06-07 14:30:00     Air leak   
3           4 2020-07-15 14:30:00 2020-07-15 19:00:00     Air leak   

      severity                    report_note        duration  
0  High stress                                0 days 23:59:00  
1  High stress  Maintenance on 30Apr at 12:00 0 days 06:30:00  
2  High stress   Maintenance on 8Jun at 16:00 2 days 04:30:00  
3  High stress  Maintenance on 16Jul at 00:00 0 days 04:30:00  


In [7]:
#Quick check
print("\nMetro variables:", metro_vars)
print("Hydraulic comparisons:")
for label, feats in hydraulic_pairs.items():
    print(f"  {label}: {feats}")


Metro variables: ['TP2', 'H1', 'DV_pressure', 'Oil_temperature', 'Motor_current']
Hydraulic comparisons:
  cooler_condition: ['TS1_mean', 'TS1_std']
  pump_leakage: ['EPS1_mean', 'PS1_mean']
  stable_flag: ['TS1_std', 'VS1_std', 'PS1_std', 'EPS1_std']


## MetroPT-3: pre-failure vs normal comparisons

The MetroPT-3 dataset contains known failure events but no direct row-level degradation labels. The main idea in this section is therefore to compare selected sensor variables between pre-failure windows and normal-operation windows.

Several candidate pre-failure window lengths may be tested in this scratchpad before one final definition is selected for the clean notebook.

### Failure-event definition

The MetroPT-3 sensor table is not labeled at the row level, but the dataset documentation provides four reported failure intervals. These are loaded as structured metadata and used to define pre-failure windows relative to the failure start time. The intervals themselves are treated as failure periods and should not be included in the normal-operation baseline.

In [9]:
# Candidate window extraction helpers

def extract_prefailure_windows(df, failure_df, hours):
    windows = []

    for _, row in failure_df.iterrows():
        ft = row['start_time']
        start = ft - pd.Timedelta(hours=hours)

        mask = (df['timestamp'] >= start) & (df['timestamp'] < ft)
        part = df.loc[mask].copy()
        part['failure_id'] = row['failure_id']
        part['failure_start'] = ft
        windows.append(part)

    return pd.concat(windows, ignore_index=True) if windows else pd.DataFrame()


def extract_normal_windows(df, failure_df, hours, buffer_hours=48):
    windows = []

    for _, row in failure_df.iterrows():
        ft = row['start_time']
        start = ft - pd.Timedelta(hours=hours + buffer_hours)
        end = ft - pd.Timedelta(hours=buffer_hours)

        mask = (df['timestamp'] >= start) & (df['timestamp'] < end)
        part = df.loc[mask].copy()
        part['failure_id'] = row['failure_id']
        part['failure_start'] = ft
        windows.append(part)

    return pd.concat(windows, ignore_index=True) if windows else pd.DataFrame()

In [11]:
# Candidate window counts

for h in [1, 6, 12, 24]:
    prefail = extract_prefailure_windows(metro, metro_failures, h)
    normal = extract_normal_windows(metro, metro_failures, h, buffer_hours=48)

    print(f"{h}h window -> pre-failure rows: {len(prefail):,}, normal rows: {len(normal):,}")

1h window -> pre-failure rows: 1,387, normal rows: 1,025
6h window -> pre-failure rows: 8,327, normal rows: 7,773
12h window -> pre-failure rows: 16,654, normal rows: 16,030
24h window -> pre-failure rows: 28,668, normal rows: 26,829


In [12]:
# Inspecting one case

prefail_6h = extract_prefailure_windows(metro, metro_failures, 6)
normal_6h = extract_normal_windows(metro, metro_failures, 6, buffer_hours=48)

print("Pre-failure 6h sample:")
print(prefail_6h[['timestamp', 'failure_id', 'failure_start']].head())

print("\nNormal 6h sample:")
print(normal_6h[['timestamp', 'failure_id', 'failure_start']].head())

Pre-failure 6h sample:
            timestamp  failure_id failure_start
0 2020-04-17 18:00:10           1    2020-04-18
1 2020-04-17 18:00:22           1    2020-04-18
2 2020-04-17 18:00:34           1    2020-04-18
3 2020-04-17 18:00:46           1    2020-04-18
4 2020-04-17 18:00:58           1    2020-04-18

Normal 6h sample:
            timestamp  failure_id failure_start
0 2020-04-15 18:00:09           1    2020-04-18
1 2020-04-15 18:00:19           1    2020-04-18
2 2020-04-15 18:00:29           1    2020-04-18
3 2020-04-15 18:00:39           1    2020-04-18
4 2020-04-15 18:00:49           1    2020-04-18


In [13]:
# Catch/Print weird imbalances

print("Pre-failure rows per event (6h):")
print(prefail_6h['failure_id'].value_counts().sort_index())

print("\nNormal rows per event (6h):")
print(normal_6h['failure_id'].value_counts().sort_index())

Pre-failure rows per event (6h):
failure_id
1    1790
2    2179
3    2179
4    2179
Name: count, dtype: int64

Normal rows per event (6h):
failure_id
1    2179
2    1786
3    2179
4    1629
Name: count, dtype: int64


In [14]:
# Find timestamp gaps inside the 6h pre-failure and normal windows

def find_gaps_in_windows(df, window_label):
    """Find gaps > 30 seconds inside extracted windows, grouped by failure_id."""
    df = df.sort_values(['failure_id', 'timestamp']).reset_index(drop=True)
    df['gap'] = df.groupby('failure_id')['timestamp'].diff()
    gaps = df[df['gap'] > pd.Timedelta(seconds=30)]
    if len(gaps) == 0:
        print(f"{window_label}: no gaps > 30s inside any window")
    else:
        print(f"{window_label}: {len(gaps)} gap(s) found")
        print(gaps[['failure_id', 'timestamp', 'gap']])

find_gaps_in_windows(prefail_6h, "Pre-failure 6h")
print()
find_gaps_in_windows(normal_6h, "Normal 6h")

Pre-failure 6h: no gaps > 30s inside any window

Normal 6h: no gaps > 30s inside any window


In [15]:
print("Pre-failure 6h actual time spans per failure:")
for fid in prefail_6h['failure_id'].unique():
    sub = prefail_6h[prefail_6h['failure_id'] == fid]
    actual_start = sub['timestamp'].min()
    actual_end = sub['timestamp'].max()
    requested_start = sub['failure_start'].iloc[0] - pd.Timedelta(hours=6)
    requested_end = sub['failure_start'].iloc[0]
    print(f"  Failure {fid}:")
    print(f"    Requested: {requested_start} -> {requested_end}")
    print(f"    Actual:    {actual_start} -> {actual_end}")
    print(f"    Coverage:  {(actual_end - actual_start).total_seconds() / 3600:.2f}h of 6h")

print("\nNormal 6h actual time spans per failure:")
for fid in normal_6h['failure_id'].unique():
    sub = normal_6h[normal_6h['failure_id'] == fid]
    actual_start = sub['timestamp'].min()
    actual_end = sub['timestamp'].max()
    requested_end = sub['failure_start'].iloc[0] - pd.Timedelta(hours=48)
    requested_start = requested_end - pd.Timedelta(hours=6)
    print(f"  Failure {fid}:")
    print(f"    Requested: {requested_start} -> {requested_end}")
    print(f"    Actual:    {actual_start} -> {actual_end}")
    print(f"    Coverage:  {(actual_end - actual_start).total_seconds() / 3600:.2f}h of 6h")

Pre-failure 6h actual time spans per failure:
  Failure 1:
    Requested: 2020-04-17 18:00:00 -> 2020-04-18 00:00:00
    Actual:    2020-04-17 18:00:10 -> 2020-04-17 23:59:49
    Coverage:  5.99h of 6h
  Failure 2:
    Requested: 2020-05-29 17:30:00 -> 2020-05-29 23:30:00
    Actual:    2020-05-29 17:30:08 -> 2020-05-29 23:29:58
    Coverage:  6.00h of 6h
  Failure 3:
    Requested: 2020-06-05 04:00:00 -> 2020-06-05 10:00:00
    Actual:    2020-06-05 04:00:06 -> 2020-06-05 09:59:54
    Coverage:  6.00h of 6h
  Failure 4:
    Requested: 2020-07-15 08:30:00 -> 2020-07-15 14:30:00
    Actual:    2020-07-15 08:30:00 -> 2020-07-15 14:29:51
    Coverage:  6.00h of 6h

Normal 6h actual time spans per failure:
  Failure 1:
    Requested: 2020-04-15 18:00:00 -> 2020-04-16 00:00:00
    Actual:    2020-04-15 18:00:09 -> 2020-04-15 23:59:59
    Coverage:  6.00h of 6h
  Failure 2:
    Requested: 2020-05-27 17:30:00 -> 2020-05-27 23:30:00
    Actual:    2020-05-27 17:30:09 -> 2020-05-27 23:29:55
   

In [16]:
for fid in [1, 2, 4]:
    print(f"\nFailure {fid} pre-failure interval distribution:")
    sub = prefail_6h[prefail_6h['failure_id'] == fid].sort_values('timestamp')
    print(sub['timestamp'].diff().value_counts().head(10))

for fid in [2, 4]:
    print(f"\nFailure {fid} normal interval distribution:")
    sub = normal_6h[normal_6h['failure_id'] == fid].sort_values('timestamp')
    print(sub['timestamp'].diff().value_counts().head(10))


Failure 1 pre-failure interval distribution:
timestamp
0 days 00:00:12    1366
0 days 00:00:13     267
0 days 00:00:11     156
Name: count, dtype: int64

Failure 2 pre-failure interval distribution:
timestamp
0 days 00:00:10    1988
0 days 00:00:09     190
Name: count, dtype: int64

Failure 4 pre-failure interval distribution:
timestamp
0 days 00:00:10    1989
0 days 00:00:09     189
Name: count, dtype: int64

Failure 2 normal interval distribution:
timestamp
0 days 00:00:12    1313
0 days 00:00:13     319
0 days 00:00:11     153
Name: count, dtype: int64

Failure 4 normal interval distribution:
timestamp
0 days 00:00:10    1484
0 days 00:00:09     143
0 days 00:00:12       1
Name: count, dtype: int64


### Window-quality note

Initial diagnostics showed that lower row counts can arise either from slightly slower regular sampling (e.g. 11–13 s intervals) or from genuinely truncated windows. Because one normal 6-hour window was substantially incomplete, later statistical comparisons should be restricted to windows that satisfy a minimum coverage threshold.

In [18]:
def filter_windows_by_coverage(df, requested_hours, min_fraction=0.9):
    coverage = (
        df.groupby('failure_id')['timestamp']
        .agg(actual_start='min', actual_end='max')
        .reset_index()
    )

    coverage['coverage_hours'] = (
        coverage['actual_end'] - coverage['actual_start']
    ).dt.total_seconds() / 3600

    coverage['keep'] = coverage['coverage_hours'] >= requested_hours * min_fraction

    keep_ids = coverage.loc[coverage['keep'], 'failure_id']
    filtered = df[df['failure_id'].isin(keep_ids)].copy()

    return filtered, coverage

In [19]:
# Apply coverage filter independently to pre-failure and normal windows
prefail_6h_filtered, prefail_coverage = filter_windows_by_coverage(
    prefail_6h, requested_hours=6
)
normal_6h_filtered, normal_coverage = filter_windows_by_coverage(
    normal_6h, requested_hours=6
)

# Keep only failures that survived both filters (pairing rule)
kept_ids = set(prefail_coverage.loc[prefail_coverage['keep'], 'failure_id']) & \
           set(normal_coverage.loc[normal_coverage['keep'], 'failure_id'])

prefail_6h_clean = prefail_6h_filtered[prefail_6h_filtered['failure_id'].isin(kept_ids)].copy()
normal_6h_clean  = normal_6h_filtered[normal_6h_filtered['failure_id'].isin(kept_ids)].copy()

print("Pre-failure coverage:")
print(prefail_coverage)

print("\nNormal coverage:")
print(normal_coverage)

print(f"\nFailure events surviving the coverage rule: {sorted(kept_ids)}")
print(f"Pre-failure rows: {len(prefail_6h_clean):,}")
print(f"Normal rows:      {len(normal_6h_clean):,}")

Pre-failure coverage:
   failure_id        actual_start          actual_end  coverage_hours  keep
0           1 2020-04-17 18:00:10 2020-04-17 23:59:49        5.994167  True
1           2 2020-05-29 17:30:08 2020-05-29 23:29:58        5.997222  True
2           3 2020-06-05 04:00:06 2020-06-05 09:59:54        5.996667  True
3           4 2020-07-15 08:30:00 2020-07-15 14:29:51        5.997500  True

Normal coverage:
   failure_id        actual_start          actual_end  coverage_hours   keep
0           1 2020-04-15 18:00:09 2020-04-15 23:59:59        5.997222   True
1           2 2020-05-27 17:30:09 2020-05-27 23:29:55        5.996111   True
2           3 2020-06-03 04:00:03 2020-06-03 09:59:54        5.997500   True
3           4 2020-07-13 08:30:06 2020-07-13 12:59:05        4.483056  False

Failure events surviving the coverage rule: [1, 2, 3]
Pre-failure rows: 6,148
Normal rows:      6,144


### Observations

The coverage filter retains three of four failure events at the 6-hour window length. Failure 4 is excluded because its paired normal window covers only 4.48 hours of the requested 6 hours, falling below the 90% coverage threshold. The pre-failure window for Failure 4 is dropped together with its normal counterpart to preserve symmetric pairing.

After filtering, the pre-failure and normal samples are closely matched in size (6,148 vs 6,144 rows), which supports a cleaner comparison between the two groups. However, because the rows within each window are temporally dependent, the next step will summarize each failure window at the event level before formal statistical testing.

The same coverage rule will be reapplied at every window length in the sensitivity sweep, and the number of retained failure events will be reported alongside the test results.

### Coverage-filter result

After applying a minimum coverage rule, only failures 1, 2, and 3 remained usable for matched 6-hour pre-failure versus normal comparison. Failure 4 was excluded because the corresponding normal window was substantially truncated.

In [20]:
# Summarize each event-window

def summarize_windows(df, variables, group_label):
    summary = (
        df.groupby('failure_id')[variables]
        .median()
        .reset_index()
    )
    summary['group'] = group_label
    return summary

In [21]:
prefail_6h_summary = summarize_windows(prefail_6h_clean, metro_vars, 'prefail_6h')
normal_6h_summary = summarize_windows(normal_6h_clean, metro_vars, 'normal_6h')

metro_event_summary_6h = pd.concat(
    [prefail_6h_summary, normal_6h_summary],
    ignore_index=True
)

metro_event_summary_6h

,failure_id,TP2,H1,DV_pressure,Oil_temperature,Motor_current,group
0,1,-0.018,8.238,-0.024,49.450,0.0400,prefail_6h
1,2,-0.012,8.764,-0.020,63.975,0.0425,prefail_6h
2,3,-0.012,8.800,-0.022,63.475,0.0450,prefail_6h
3,1,-0.014,8.886,-0.024,57.725,0.0425,normal_6h
4,2,-0.010,8.158,-0.018,64.600,0.0400,normal_6h
5,3,-0.012,8.864,-0.022,63.775,0.0425,normal_6h


In [22]:
# Checking whether the per-event medians move in the same direction

for var in metro_vars:
    print(f"\n{var}")
    print(
        metro_event_summary_6h[['failure_id', 'group', var]]
        .sort_values(['failure_id', 'group'])
        .to_string(index=False)
    )


TP2
 failure_id      group    TP2
          1  normal_6h -0.014
          1 prefail_6h -0.018
          2  normal_6h -0.010
          2 prefail_6h -0.012
          3  normal_6h -0.012
          3 prefail_6h -0.012

H1
 failure_id      group    H1
          1  normal_6h 8.886
          1 prefail_6h 8.238
          2  normal_6h 8.158
          2 prefail_6h 8.764
          3  normal_6h 8.864
          3 prefail_6h 8.800

DV_pressure
 failure_id      group  DV_pressure
          1  normal_6h       -0.024
          1 prefail_6h       -0.024
          2  normal_6h       -0.018
          2 prefail_6h       -0.020
          3  normal_6h       -0.022
          3 prefail_6h       -0.022

Oil_temperature
 failure_id      group  Oil_temperature
          1  normal_6h           57.725
          1 prefail_6h           49.450
          2  normal_6h           64.600
          2 prefail_6h           63.975
          3  normal_6h           63.775
          3 prefail_6h           63.475

Motor_current
 

### Observations

After summarizing the retained 6-hour windows at the failure-event level, the clearest and most consistent pre-failure shift appears in `Oil_temperature`: all three retained failures show lower median oil temperature in the pre-failure window than in the matched normal window. `H1` also shows a consistent downward shift across the three events.

`TP2` shows a weaker pattern, with a pre-failure decrease in two of the three failures but no change in the third. By contrast, `DV_pressure` shows almost no systematic difference, and `Motor_current` changes in inconsistent directions across events.

At the 6-hour window level, the most promising MetroPT-3 candidates for formal comparison therefore appear to be `Oil_temperature` and `H1`, with `TP2` as a weaker secondary candidate.

## Tomorrow's work
- Final MetroPT window choice: **6h** (12h as sensitivity check)
- Final MetroPT kept variables: **Primary: `Oil_temperature`, `TP2`. Secondary: `H1`. Dropped: `Motor_current`, `DV_pressure`.**
- Final hydraulic kept tests:
- Reliability section scope:
- Items ready to move into clean Notebook 03:

In [23]:
# Window-length sweep: extract, filter by coverage, compute per-event medians

WINDOW_LENGTHS_HOURS = [1, 6, 12, 24]
COVERAGE_THRESHOLD = 0.9

sweep_records = []
sweep_kept_ids = {}  # window_length -> kept failure_ids

for h in WINDOW_LENGTHS_HOURS:
    # Extract windows
    prefail = extract_prefailure_windows(metro, metro_failures, h)
    normal  = extract_normal_windows(metro, metro_failures, h, buffer_hours=48)

    # Apply coverage filter (per-half), then enforce pairing
    prefail_filt, prefail_cov = filter_windows_by_coverage(
        prefail, requested_hours=h, min_fraction=COVERAGE_THRESHOLD
    )
    normal_filt, normal_cov = filter_windows_by_coverage(
        normal, requested_hours=h, min_fraction=COVERAGE_THRESHOLD
    )
    kept_ids = sorted(
        set(prefail_cov.loc[prefail_cov['keep'], 'failure_id']) &
        set(normal_cov.loc[normal_cov['keep'], 'failure_id'])
    )
    sweep_kept_ids[h] = kept_ids

    # Filter both halves to surviving failures
    prefail_clean = prefail_filt[prefail_filt['failure_id'].isin(kept_ids)]
    normal_clean  = normal_filt[normal_filt['failure_id'].isin(kept_ids)]

    # Per-event medians, long format
    for fid in kept_ids:
        for var in metro_vars:
            pre_median  = prefail_clean.loc[
                prefail_clean['failure_id'] == fid, var
            ].median()
            norm_median = normal_clean.loc[
                normal_clean['failure_id'] == fid, var
            ].median()

            sweep_records.append({
                'window_h': h,
                'failure_id': fid,
                'variable': var,
                'prefail_median': pre_median,
                'normal_median':  norm_median,
                'difference':     pre_median - norm_median,
            })

sweep_df = pd.DataFrame(sweep_records)

# Quick summary: how many failures survived at each window length
print("Failures retained per window length:")
for h in WINDOW_LENGTHS_HOURS:
    print(f"  {h:>2}h: {sweep_kept_ids[h]}")

print(f"\nTotal sweep records: {len(sweep_df)}")
print("\nSweep dataframe head:")
print(sweep_df.head(15))

Failures retained per window length:
   1h: [1, 2, 3]
   6h: [1, 2, 3]
  12h: [1, 2, 3]
  24h: [1, 2, 3]

Total sweep records: 60

Sweep dataframe head:
    window_h  n_kept_failures  failure_id         variable  prefail_median  \
0          1                3           1              TP2          -0.018   
1          1                3           1               H1           8.238   
2          1                3           1      DV_pressure          -0.024   
3          1                3           1  Oil_temperature          49.450   
4          1                3           1    Motor_current           0.040   
5          1                3           2              TP2          -0.012   
6          1                3           2               H1           8.450   
7          1                3           2      DV_pressure          -0.020   
8          1                3           2  Oil_temperature          63.650   
9          1                3           2    Motor_current         

**For each variable, at which window length is the direction consistency strongest, and is the median magnitude meaningful?**

In [24]:
consistency_summary = (
    sweep_df.groupby(['window_h', 'variable'])['prefail_minus_normal']
    .agg(
        median_diff='median',
        n_negative=lambda x: (x < 0).sum(),
        n_positive=lambda x: (x > 0).sum(),
        n_zero=lambda x: (x == 0).sum()
    )
    .reset_index()
)

consistency_summary


,window_h,variable,median_diff,n_negative,n_positive,n_zero
0,1,DV_pressure,0.0000,1,0,2
1,1,H1,-0.4580,2,1,0
2,1,Motor_current,0.0025,1,2,0
3,1,Oil_temperature,-0.9500,2,1,0
4,1,TP2,-0.0020,2,0,1
5,6,DV_pressure,0.0000,1,0,2
6,6,H1,-0.0640,2,1,0
7,6,Motor_current,0.0025,1,2,0
8,6,Oil_temperature,-0.6250,3,0,0
9,6,TP2,-0.0020,2,0,1


In [25]:
# Sorting
consistency_summary.sort_values(['variable', 'window_h'])

,window_h,variable,median_diff,n_negative,n_positive,n_zero
0,1,DV_pressure,0.0000,1,0,2
5,6,DV_pressure,0.0000,1,0,2
10,12,DV_pressure,0.0000,1,0,2
15,24,DV_pressure,0.0000,0,0,3
1,1,H1,-0.4580,2,1,0
6,6,H1,-0.0640,2,1,0
11,12,H1,-0.0120,2,1,0
16,24,H1,0.0600,1,2,0
2,1,Motor_current,0.0025,1,2,0
7,6,Motor_current,0.0025,1,2,0


In [26]:
# Consistency count

# Direction summary string per (variable, window_h)
consistency_summary['direction_summary'] = (
    consistency_summary['n_negative'].astype(str) + 'n / ' +
    consistency_summary['n_positive'].astype(str) + 'p / ' +
    consistency_summary['n_zero'].astype(str) + 'z'
)

direction_pivot = consistency_summary.pivot(
    index='variable',
    columns='window_h',
    values='direction_summary'
)
direction_pivot

window_h,1,6,12,24
variable,,,,
DV_pressure,1n / 0p / 2z,1n / 0p / 2z,1n / 0p / 2z,0n / 0p / 3z
H1,2n / 1p / 0z,2n / 1p / 0z,2n / 1p / 0z,1n / 2p / 0z
Motor_current,1n / 2p / 0z,1n / 2p / 0z,1n / 2p / 0z,1n / 2p / 0z
Oil_temperature,2n / 1p / 0z,3n / 0p / 0z,2n / 1p / 0z,1n / 2p / 0z
TP2,2n / 0p / 1z,2n / 0p / 1z,1n / 0p / 2z,1n / 0p / 2z


### Observations

The window-length sweep suggests that the 6-hour pre-failure window provides the clearest and most interpretable MetroPT-3 signal. `Oil_temperature` shows the strongest consistency at this window length, with all three retained failures exhibiting lower pre-failure medians than their matched normal windows. `H1` shows a weaker but still partly consistent downward shift across the shorter windows.

By contrast, `Motor_current` shows mixed direction across events at every tested window length, indicating that its variation reflects operating-regime changes rather than a pre-failure signature. `DV_pressure` shows almost no systematic change between pre-failure and normal windows at any window length and shows no observable pre-failure signal in this analysis. `TP2` shows a partial signal (two of three failures with lower pre-failure medians, the third flat), placing it as a weaker but coherent secondary candidate. `H1` shows partial consistency at the shorter windows but with a direction flip in one failure, weakening its interpretation.

Based on this sweep, the 6-hour window will be treated as the main MetroPT-3 comparison window, while 12 hours may be retained as a sensitivity check if needed.

### Event-level Mann-Whitney U at the 6-hour window

The formal hypothesis tests for MetroPT-3 are conducted at the event-summary level, using the per-failure median as the unit of comparison rather than individual sensor rows. This protects against the within-window autocorrelation that would inflate row-level significance, but reduces sample size to n=3 vs n=3 after the coverage filter, which limits the statistical power available.

Three variables are tested: `Oil_temperature`, `TP2`, and `H1`. `Motor_current` and `DV_pressure` were excluded based on the window-length sweep findings reported above. Bonferroni correction across $k=3$ tests sets the corrected significance threshold at $\alpha = 0.05/3 \approx 0.0167$.

Per the planned interpretation rule, results are evaluated through the combination of significance, direction, magnitude, and physical plausibility, with significance treated as one criterion among several rather than as the decisive measure.

In [29]:
def mannwhitney_table(df_a, df_b, variables, group_a='A', group_b='B'):
    """Mann-Whitney U for each variable, with Bonferroni-corrected
    p-values, rank-biserial effect size, sample sizes, and direction."""
    results = []
    for var in variables:
        x = df_a[var].dropna()
        y = df_b[var].dropna()

        stat, p = mannwhitneyu(x, y, alternative='two-sided')

        n1, n2 = len(x), len(y)
        rank_biserial = 1 - (2 * stat) / (n1 * n2)

        diff = x.median() - y.median()
        if diff > 0:
            direction = f'{group_a} > {group_b}'
        elif diff < 0:
            direction = f'{group_b} > {group_a}'
        else:
            direction = 'equal'

        results.append({
            'variable':           var,
            f'n_{group_a}':       n1,
            f'n_{group_b}':       n2,
            f'{group_a}_median':  x.median(),
            f'{group_b}_median':  y.median(),
            'direction':          direction,
            'u_stat':             stat,
            'p_value':            p,
            'rank_biserial':      rank_biserial,
        })

    out = pd.DataFrame(results)
    out['p_bonf'] = multipletests(out['p_value'], method='bonferroni')[1]
    return out.sort_values('p_value')

In [30]:
metro_vars_final_6h = ['Oil_temperature', 'TP2', 'H1']

In [31]:
metro_event_results_6h = mannwhitney_table(
    prefail_6h_summary,
    normal_6h_summary,
    metro_vars_final_6h,
    group_a='prefail',
    group_b='normal'
)

metro_event_results_6h

,variable,n_prefail,n_normal,prefail_median,normal_median,direction,u_stat,p_value,rank_biserial,p_bonf
1,TP2,3,3,-0.012,-0.012,equal,3.0,0.642835,0.333333,1.0
0,Oil_temperature,3,3,63.475,63.775,normal > prefail,3.0,0.700000,0.333333,1.0
2,H1,3,3,8.764,8.864,normal > prefail,3.0,0.700000,0.333333,1.0


### Observations

In [ ]:
### Observations

The event-level Mann-Whitney U results for the retained 6-hour MetroPT-3 windows do not reach statistical significance after Bonferroni correction. This is expected, given that only three matched failure events remain available after the coverage filter, which makes formal inference very low-powered.

Even so, the direction of the event-level medians remains informative. `Oil_temperature` and `H1` both show lower pre-failure medians than their matched normal windows, consistent with the earlier descriptive inspection. By contrast, `TP2` does not show a clear difference at the grouped median level, even though the per-event view showed lower pre-failure medians in two of the three retained failures. This illustrates that the median-of-three event summary is a stricter statistic than simple direction counting and may hide partial shifts that are not consistent across most events.

At this stage, the MetroPT-3 evidence is therefore better interpreted as directional and exploratory-supportive rather than as strong standalone statistical confirmation. In the final notebook, the 6-hour window can still be retained as the main MetroPT comparison, but the limited number of matched failure events must be stated explicitly when discussing the test results.